In [1]:
import socket
import re
from urllib.request import urlopen

def discover_tvs(timeout=3):
    """Discovers TVs on the same Wi-Fi network using SSDP."""
    SSDP_GROUP = ("239.255.255.250", 1900)
    
    # Common UPnP targets for TVs
    st_targets = [
        "urn:schemas-upnp-org:device:MediaRenderer:1",
        "urn:dial-multiscreen-org:service:dial:1" # Used by Netflix/YouTube on Smart TVs
    ]

    discovered_tvs = {}

    for st in st_targets:
        message = (
            "M-SEARCH * HTTP/1.1\r\n"
            "HOST: 239.255.255.250:1900\r\n"
            "MAN: \"ssdp:discover\"\r\n"
            f"ST: {st}\r\n"
            "MX: 2\r\n"
            "\r\n"
        )

        sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM, socket.IPPROTO_UDP)
        sock.settimeout(timeout)

        try:
            sock.sendto(message.encode('utf-8'), SSDP_GROUP)
            while True:
                data, addr = sock.recvfrom(2048)
                response = data.decode('utf-8', errors='ignore')
                
                if 'LOCATION' in response.upper():
                    # Extract location URL
                    location = next((line.split(':', 1)[1].strip() for line in response.split('\r\n') if line.upper().startswith('LOCATION:')), None)
                    if location:
                        ip = addr[0]
                        if ip not in discovered_tvs:
                            discovered_tvs[ip] = location
        except socket.timeout:
            pass
        finally:
            sock.close()

    return discovered_tvs

def fetch_tv_info(location_url):
    """Tries to fetch basic info about the TV from its XML descriptor."""
    try:
        with urlopen(location_url, timeout=2) as response:
            xml_data = response.read().decode('utf-8')
            
            friendly_name_match = re.search(r'<friendlyName>(.*?)</friendlyName>', xml_data)
            manufacturer_match = re.search(r'<manufacturer>(.*?)</manufacturer>', xml_data)
            model_name_match = re.search(r'<modelName>(.*?)</modelName>', xml_data)
            
            name = friendly_name_match.group(1) if friendly_name_match else "Unknown TV"
            manufacturer = manufacturer_match.group(1) if manufacturer_match else "Unknown"
            model = model_name_match.group(1) if model_name_match else "Unknown"
            
            return f"{name} ({manufacturer} {model})"
    except Exception:
        return "Unknown Smart TV"



In [2]:
print("Searching for TVs on the network... (this will take a few seconds)")
tvs = discover_tvs()

if not tvs:
    print("No TVs found. Make sure you are on the same Wi-Fi network as your TV.")
else:
    print(f"\nDiscovered {len(tvs)} TV(s):")
    for ip, location in tvs.items():
        info = fetch_tv_info(location)
        print(f"- IP: {ip} | Info: {info} | Location: {location}")


Searching for TVs on the network... (this will take a few seconds)

Discovered 2 TV(s):
- IP: 192.168.29.13 | Info: Unknown Smart TV | Location: http://192.168.29.13:9197/dmr
- IP: 192.168.29.158 | Info: Family room clock (LENOVO Lenovo Smart Clock) | Location: http://192.168.29.158:8008/ssdp/device-desc.xml


In [5]:
import asyncio
import json
import websockets

TV_IP = "192.168.29.232"

async def pair_tv():
    uri = f"ws://{TV_IP}:3000/"
    register_payload = {
        "type": "register",
        "id": "register_0",
        "payload": {
            "forcePairing": False,
            "pairingType": "PROMPT",
            "manifest": {
                "manifestVersion": 1,
                "appVersion": "1.0",
                "permissions": [
                    "LAUNCH",
                    "CONTROL_AUDIO",
                    "CONTROL_POWER",
                    "READ_NETWORK_STATE"
                ]
            }
        }
    }

    try:
        print(f"Attempting to connect to TV at {uri}...")
        async with websockets.connect(uri, ping_interval=None) as websocket:
            print("Connected! Sending pairing request...")
            print("\n*** PLEASE LOOK AT YOUR TV SCREEN AND ACCEPT THE PROMPT ***\n")
            
            await websocket.send(json.dumps(register_payload))
            
            while True:
                response = await websocket.recv()
                resp_data = json.loads(response)
                
                if resp_data.get("type") == "registered":
                    client_key = resp_data.get("payload", {}).get("client-key")
                    print(f"SUCCESS! Paired with TV.")
                    print(f"Your CLIENT KEY is: {client_key}")
                    print("Save this key to use in your other scripts/apps!")
                    break
                elif resp_data.get("type") == "error":
                    print(f"Error from TV: {resp_data.get('error')}")
                    break
                else:
                    print(f"Waiting for you to accept on TV... (Received message type: {resp_data.get('type')})")
                    
    except Exception as e:
        print(f"Connection failed: {e}")
        print("Note: If port 3000 fails, your TV might require secure websockets on port 3001.")

await pair_tv()


Attempting to connect to TV at ws://192.168.29.232:3000/...
Connection failed: did not receive a valid HTTP response
Note: If port 3000 fails, your TV might require secure websockets on port 3001.
